### 1. Import Libraries

This cell imports the `kagglehub` library, which is a convenient tool for downloading datasets directly from Kaggle within a Colab environment.

In [ ]:
import kagglehub

### 2. Download Dataset

This cell uses `kagglehub.dataset_download` to fetch the 'fruit-ripeness-unripe-ripe-and-rotten' dataset. The returned `path` variable holds the local directory where the dataset has been downloaded and extracted.

In [ ]:
path=kagglehub.dataset_download("leftin/fruit-ripeness-unripe-ripe-and-rotten")

100%|██████████| 3.63G/3.63G [00:59<00:00, 65.4MB/s]

Extracting files...


### 3. Verify Download Path

This cell simply prints the `path` variable to confirm where the dataset files are located on the system.

In [ ]:
print(path)

/root/.cache/kagglehub/datasets/leftin/fruit-ripeness-unripe-ripe-and-rotten/versions/2


### 4. File System Operations

The `os` module provides a way to interact with the operating system, including file and directory operations. This import is necessary for tasks like listing directory contents.

In [ ]:
import os

### 5. List Dataset Contents

This cell uses `os.listdir(path)` to show the immediate subdirectories or files within the main downloaded dataset folder. This helps to understand the top-level structure of the extracted data.

In [ ]:
print(os.listdir(path))

['fruit_ripeness_dataset']


This cell is a repeat of the earlier `print(path)` statement. It serves as another check for the base download directory.

### 9. PyTorch and Torchvision Imports

In [49]:
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms, datasets , models
import torch.optim as optim
import torch

This cell defines a `transform` pipeline using `torchvision.transforms.Compose`. This sequence of operations will be applied to each image:
- `Resize((224,224))`: Resizes images to a standard 224x224 pixel dimension, a common input size for many pre-trained convolutional neural networks.
- `ToTensor()`: Converts images from PIL Image format (or NumPy arrays) to PyTorch tensors.
- `Normalize(...)`: Standardizes the pixel values across the R, G, B channels using mean and standard deviation derived from the ImageNet dataset. This helps in training deep learning models more effectively.

### 6. Verify Base Path (Redundant Check)

In [ ]:
print(path)

/root/.cache/kagglehub/datasets/leftin/fruit-ripeness-unripe-ripe-and-rotten/versions/2


### 7. Inspect Training Data Directory

This cell executes a shell command (`!dir`) to list the contents of the specific `train` directory within the dataset. This helps confirm the presence of class-specific subfolders (e.g., 'freshapples', 'rottenbanana') which is typical for image classification datasets.

In [ ]:
!dir /root/.cache/kagglehub/datasets/leftin/fruit-ripeness-unripe-ripe-and-rotten/versions/2/fruit_ripeness_dataset/archive\ \(1\)/dataset/train

freshapples  freshoranges  rottenbanana   unripe\ apple   unripe\ orange
freshbanana  rottenapples  rottenoranges  unripe\ banana


### 8. Define Training Directory Path (Original Attempt)

This cell defined the `dir` variable using a raw string literal to specify the path to the training data. The manual escaping (`\ ` and `\(`) for spaces and parentheses, while necessary for shell commands, caused issues when interpreted by Python's `os.scandir` in later steps, leading to `FileNotFoundError`. This variable is superseded by `correct_train_path` which uses `os.path.join` for robustness.

In [ ]:
dir=r"/root/.cache/kagglehub/datasets/leftin/fruit-ripeness-unripe-ripe-and-rotten/versions/2/fruit_ripeness_dataset/archive\ \(1\)/dataset/train"

### 10. Define Image Transformations

This cell imports all the necessary modules from PyTorch (`torch`), its neural network module (`torch.nn`), data utilities (`torch.utils.data.DataLoader`), and computer vision utilities (`torchvision.transforms`, `torchvision.datasets`, `torchvision.models`). `torch.optim` is also imported for defining optimizers.

In [ ]:
transform=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


### 11. Create Dataset Object

This cell creates a `torchvision.datasets.ImageFolder` object. This class is designed to handle image datasets organized with subfolders representing different classes. It automatically infers class labels from folder names.
- `correct_train_path`: The dynamically constructed, robust path to the 'train' directory.
- `transform=transform`: Applies the predefined transformations to each image loaded from the dataset.

In [ ]:
import os

# Reconstruct the correct path using os.path.join with the base 'path' and subdirectories.
# This ensures spaces and parentheses are handled correctly.
correct_train_path = os.path.join(path, 'fruit_ripeness_dataset', 'archive (1)', 'dataset', 'train')

dataset=datasets.ImageFolder(correct_train_path,transform=transform)

### 12. Create DataLoader

This cell creates a `DataLoader` from the `dataset` object. The `DataLoader` is responsible for feeding data to the neural network in mini-batches during training, handling shuffling and parallel data loading.
- `batch_size=32`: Specifies that 32 images will be processed at once.
- `shuffle=True`: Ensures the data is randomly shuffled at the beginning of each epoch, which helps improve model generalization.

In [ ]:
train_loader=DataLoader(dataset,batch_size=32,shuffle=True)

### 13. Load Pre-trained Model

This cell loads a pre-trained MobileNetV3-Small model from `torchvision.models`. `weights=models.MobileNet_V3_Small_Weights.DEFAULT` initializes the model with weights trained on the vast ImageNet dataset. This is a common strategy in transfer learning, leveraging powerful features learned from a large dataset for a new, related task.

In [ ]:
model=models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)

Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:00<00:00, 64.5MB/s]


### 14. Adapt Classifier for New Task

This cell modifies the final classification layer of the pre-trained MobileNetV3 model. Since the original model was designed for ImageNet (1000 classes), its final layer needs to be replaced with a new linear layer (`nn.Linear`) that outputs the number of classes specific to our fruit ripeness dataset (`len(dataset.classes)`). This customization is a key step in fine-tuning a pre-trained model for a new task.

In [ ]:
model.classifier[3]=nn.Linear(model.classifier[3].in_features,len(dataset.classes))

### 15. Define Loss Function

This cell defines the `criterion` (loss function) as `nn.CrossEntropyLoss()`. This is a standard loss function for multi-class classification problems in PyTorch. It combines `LogSoftmax` and `NLLLoss` to calculate the difference between the model's predictions and the true labels.

In [ ]:
criterion=nn.CrossEntropyLoss()

### 16. Define Optimizer

This cell defines the `optimizer` that will be used to update the model's weights during training. `torch.optim.Adam` is a popular and effective optimization algorithm. It takes the model's parameters (`model.parameters()`) to optimize and a `learning rate (lr=0.001)`, which controls the step size of each update.

In [ ]:
optimizer=torch.optim.Adam(model.parameters(),lr=0.001)

In [50]:
model.train()


MobileNetV3(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
      (2): Hardswish()
    )
    (1): InvertedResidual(
      (block): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=16, bias=False)
          (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
          (2): ReLU(inplace=True)
        )
        (1): SqueezeExcitation(
          (avgpool): AdaptiveAvgPool2d(output_size=1)
          (fc1): Conv2d(16, 8, kernel_size=(1, 1), stride=(1, 1))
          (fc2): Conv2d(8, 16, kernel_size=(1, 1), stride=(1, 1))
          (activation): ReLU()
          (scale_activation): Hardsigmoid()
        )
        (2): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(1, 1), 

In [ ]:
for images, labels in train_loader:
  optimizer.zero_grad()
  outputs=model(images)
  loss=criterion(outputs,labels)
  loss.backward()
  optimizer.step()


### 17. Testing Phase

This cell defines the path to the test dataset and creates an `ImageFolder` object for it, applying the same transformations as the training data. Then, a `DataLoader` is created for the test set, with shuffling set to `False` as it's for evaluation.

In [51]:
correct_test_path = os.path.join(path, 'fruit_ripeness_dataset', 'archive (1)', 'dataset', 'test')
test_dataset = datasets.ImageFolder(correct_test_path, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

This cell implements the testing loop. The model is set to evaluation mode (`model.eval()`) to disable dropout and batch normalization updates. It then iterates through the `test_loader`, makes predictions, calculates the loss, and tracks the number of correct predictions to compute the overall test accuracy.

In [52]:
model.eval()  # Set the model to evaluation mode
test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():  # Disable gradient calculation during testing
    for images, labels in test_loader:
        outputs = model(images)
        loss = criterion(outputs, labels)
        test_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Test Loss: {test_loss / len(test_loader):.4f}')
print(f'Test Accuracy: {100 * correct / total:.2f}%')

Test Loss: 0.1410
Test Accuracy: 95.59%


### 18. Predict on Your Own Image

This section allows you to upload an image from your local machine and use the trained model to predict its fruit ripeness category. Follow the steps below:

#### Upload an Image

Run the cell below to open a file uploader. Select the image you wish to classify.

In [62]:
from google.colab import files
from PIL import Image
import io

# Upload an image
uploaded = files.upload()

# Get the filename of the uploaded image
for fn in uploaded.keys():
    print('User uploaded file "{name}" with length {length} bytes'.format(name=fn, length=len(uploaded[fn])))
    image_filename = fn

Saving 1771269749791.webp to 1771269749791.webp
User uploaded file "1771269749791.webp" with length 136052 bytes


#### Preprocess and Predict

This cell loads the uploaded image, applies the same transformations used for the training data, and then uses the trained model to make a prediction. The predicted class (fruit ripeness category) will be printed.

In [63]:
if 'image_filename' in locals():
    # Load the image
    image = Image.open(io.BytesIO(uploaded[image_filename]))

    # Apply the same transformations as the training data
    # Ensure 'transform' is defined from earlier cells
    input_tensor = transform(image).unsqueeze(0)  # Add batch dimension

    # Set model to evaluation mode
    model.eval()

    # Make prediction
    with torch.no_grad():
        output = model(input_tensor)

    # Get predicted class
    _, predicted_idx = torch.max(output, 1)

    # Get class names from the dataset.classes attribute
    class_names = dataset.classes # assuming 'dataset' is available from earlier cells
    predicted_class = class_names[predicted_idx.item()]

    print(f"The uploaded image is predicted to be: {predicted_class}")

    # Optional: Display the image
    # from IPython.display import display
    # display(image)
else:
    print("No image was uploaded. Please run the upload cell above.")

The uploaded image is predicted to be: unripe orange


### 19. Save the Trained Model

This cell saves the trained model's state dictionary. Saving the state dictionary is a common practice in PyTorch, as it allows you to save and load only the learned parameters, making the file size smaller and more flexible for deployment or further training.

In [57]:
# Define the path where the model will be saved
model_save_path = 'fruit_ripeness_model.pth'

# Save the model's state dictionary
torch.save(model.state_dict(), model_save_path)

print(f"Model saved successfully to {model_save_path}")

Model saved successfully to fruit_ripeness_model.pth


### 20. Download the Saved Model

This cell provides an option to download the `fruit_ripeness_model.pth` file to your local machine. This allows you to use the trained model elsewhere or keep a local backup.

In [58]:
from google.colab import files

# Download the saved model file
files.download(model_save_path)

print(f"Downloading {model_save_path}...")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>